# Imports

In [1]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json

from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api()

Request timeout from https://tdp.mitre.org.


ReadTimeout: HTTPSConnectionPool(host='tdp.mitre.org', port=443): Read timed out. (read timeout=30)

Set configuration parameters to better optimize queries.

In [ ]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [ ]:
api.start_spark(n_executors=400, config=config)

# Define global variables and functions

In [ ]:
airports = [
    "KADW",
    "KATL",
    "KBOS",
    "KBWI",
    "KCLT",
    "KDCA",
    "KDEN",
    "KDFW",
    "KDTW",
    "KEWR",
    "KFLL",
    "KIAD",
    "KIAH",
    "KJFK",
    "KLAS",
    "KLAX",
    "KLGA",
    "KMCO",
    "KMDW",
    "KMEM",
    "KMIA",
    "KMSP",
    "KORD",
    "KPHL",
    "KPHX",
    "KSAN",
    "KSDF",
    "KSEA",
    "KSFO",
    "KSLC",
    "KTPA",
    "PANC",
    "PHNL",
]

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [ ]:
year0 = "2019"
year1 = str(int(year0) + 1)

In [ ]:
dates = {"start_date": year0 + "-10-01", "end_date": year1 +"-01-01"}

# Retrieve all north american procedure fixes

## retrieve procedure fixes of interest
<li>icao_code = K, CY, PA, PH, TS (continental US, canada, alaska, hawaii, and puerto rico)</li>
<li>only get SID and STAR procedure types. (approach type is ignored)</li>
<li>create a unique <i>grouping</i> column for each fix.</li>

this will retrieve multiple versions of each fix, one for each update cycle in the year.

In [ ]:
df_procedure_fixes = (
    api.dataframe("ArincTransition", **dates, metadata=True)
    .select(
#        F.col("airport.name").alias("arpt"),
        F.col("airport.icao_region").alias("icao_region"),
        "procedure_name",
        "procedure_type",
        F.explode("legs").alias("leg"),
        F.col("leg.path_terminator.identification.name").alias("fix_name"),
        F.col("leg.path_terminator.latitude"),
        F.col("leg.path_terminator.longitude"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
    .withColumn("grouping", F.concat("fix_name", F.lit("_"), "procedure_name", F.lit("_"), "icao_region"))
    .filter(F.col("fix_name").isNotNull())
    .filter(F.col("icao_region").startswith("K") | F.col("icao_region").startswith("CY") | F.col("icao_region").startswith("PA") | F.col("icao_region").startswith("PH") | F.col("icao_region").startswith("TS"))
    .filter(F.col("procedure_type").isin("SID", "STAR"))
#    .filter(F.col("arpt").isin(airports))
    .drop("leg")
    .orderBy("grouping")
)

#df_procedure_fixes.show()

## keep the newest update for each fix

In [ ]:
window = Window.partitionBy("grouping").orderBy(col("end_date").desc())

df_procedure_fixes_newest = (df_procedure_fixes
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "grouping", "end_date")
    .orderBy("fix_name")
)

#df_procedure_fixes_newest.show()

## create the output
<li>add a column that concats all the procedures that use the fix</li>
<li>add a column that concats the procedure types that use the fix</li>

In [ ]:
(
    df_procedure_fixes_newest.groupBy("fix_name", "icao_region", "latitude", "longitude")
    .agg(F.concat_ws(":", F.collect_set("procedure_name")).alias("procedures_using_fix"), F.concat_ws(":", F.collect_set("procedure_type")).alias("procedure_types_using_fix"))
#    .agg(F.concat_ws(":", F.collect_set("procedure_name")).alias("procedures_fix"))
    .orderBy("fix_name")
    .write.option("header", True)
    .csv("CRAFT/" + year0 + "/procedures", compression="None", mode="overwrite")
)

# Retrieve all US SIDs and STARs

## retrieve US procedures
<li>icao_code = K (continental US, canada, alaska, hawaii, and puerto rico)</li>
<li>only get SID and STAR procedure types. (approach type is ignored)</li>
<li>create a unique <i>grouping</i> column for each fix.</li>

this will retrieve multiple versions of each fix, one for each update cycle in the year.

In [ ]:
df_procedures = (
    api.dataframe("ArincTransition", **dates, metadata=True)
    .select(
        "dtpp_procedure_name",
        "procedure_name",
        "procedure_type",
        "transition_name",
        "transition_type",
        F.col("airport.icao_region").alias("icao_region"),
        F.col("airport.name").alias("airport"),
        F.explode("legs").alias("leg"),
        F.col("leg.path_terminator.identification.name").alias("fix_name"),
        F.col("leg.sequence_number").alias("seq"),
        F.col("leg.path_terminator.latitude"),
        F.col("leg.path_terminator.longitude"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
    .withColumn("grouping", F.concat("procedure_name", F.lit("_"), "transition_name", F.lit("_"), "seq"))
    .filter(F.col("transition_name").isNotNull())
    .filter(F.col("fix_name").isNotNull())
    .filter(F.col("icao_region").startswith("K") | F.col("icao_region").startswith("PA") | F.col("icao_region").startswith("PH") | F.col("icao_region").startswith("TS"))
    .filter(F.col("procedure_type").isin("SID", "STAR"))
    .filter(F.col("airport").isin(airports))
#    .filter(F.col("facilityID") == "KDCA")
    .drop("leg")   
    .orderBy("grouping")
)
#df_procedures.show()

## keep the newest update for each fix

In [ ]:
window = Window.partitionBy("grouping").orderBy(col("end_date").desc())

df_procedure_newest = (df_procedures
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "end_date")
    .orderBy("grouping")
)
#df_procedure_newest.show()

## create the output

In [ ]:
(
    df_procedure_newest
        .repartition("airport")
        .select(
            "dtpp_procedure_name",
            "procedure_name",
            "procedure_type",
            "transition_name",
            "transition_type",
            "fix_name",
            "seq",
            "latitude",
            "longitude",
            "airport",
        )
        .orderBy("grouping")
        .write.option("header",True).partitionBy(["airport"])
        .csv("CRAFT/" + year0 + "/procedures", compression="None", mode="overwrite")
)